# Experiment 1 - Zero-Shot Cross-Lingual Transfer (EN -> BN)

Fine-tune on **English** splits and evaluate zero-shot on the **native Bengali** test split.

| Condition | Backbone | Train | Val | Test |
|---|---|---|---|---|
| **1** zero-shot | `FacebookAI/xlm-roberta-base` | en-train | en-val | bn-test |

Uses the **same backbone** (`FacebookAI/xlm-roberta-base`), training recipe, and held-out
Bengali test split as other experiments, isolating the effect of zero-shot cross-lingual transfer from English training data.

Run with 3 seeds (42, 43, 44).
Metrics: macro-F1 (primary), per-class F1, QWK, MAE, and the
upgrade/ok/downgrade error-direction matrix.

`transformers==5.18.0` is already the version Colab's current runtime ships,
so the install cell below is a no-op and nothing preinstalled is disturbed.

**Uses GitHub Branch "experiment_zeroshot"**

**Models are pushed to a private HF repo after every run** "HF_REPO = "simonfrnd/nlp_experiment1""

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch, transformers

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device       :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU found. In Colab go to Runtime > Change runtime type > "
        "Hardware accelerator = GPU, then re-run this cell."
    )

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's
# CUDA build stays intact.
!pip install -q "transformers==5.18.0" accelerate openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(
        ["git", "clone", "--branch", "experiment_zeroshot", "--depth", "1", "https://github.com/iammatthi/NLP.git"],
        check=True,
    )
os.chdir("NLP")
print("working dir:", os.getcwd())

for rel in (
    "data/raw/depression-severity-english.csv",
    "data/raw/depression-severity-bengali.xlsx",
):
    data_file = pathlib.Path(rel)
    assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
    print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# Persist trained checkpoints to a private Hugging Face Hub repo.
# Set HF_TOKEN in Colab Secrets
import os

HF_REPO = "simonfrnd/nlp_experiment1"   # experiment 1 writes to this model repo
print("HF_REPO =", HF_REPO)

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN: read from Colab Secrets")
except Exception:
    print("HF_TOKEN: no Colab Secret found, falling back to a login prompt")

from huggingface_hub import login, whoami
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()

print("authenticated as:", whoami()["name"])
print("keep the repo private: the source datasets state no licence")

In [ ]:
# CPU-only build check: verifies the backbone loads and produces (n, 4) logits.
# Exits non-zero on failure.
!python scripts/smoke_test_model.py --models FacebookAI/xlm-roberta-base

In [ ]:
# Stratified 80/10/10 splits, fixed seed. Regenerated every session; the seed
# makes them byte-reproducible so runs stay comparable.
# Both languages are needed: en for training/validation, bn for zero-shot evaluation.
!python scripts/prepare_splits.py --lang en --seed 42
!python scripts/prepare_splits.py --lang bn --seed 42

## Train

Class-weighted cross-entropy is on by default (inverse-frequency from the English
train split) to counteract the 73%-minimal label prior without resampling. Checkpoints
are selected on English validation macro-F1; the best one is written to `<output-dir>/final/`
and pushed to the Hub immediately.

In [ ]:
%%bash
set -e
# Condition 1 - XLM-R on English (zero-shot cross-lingual transfer)
python train.py --model xlmr --lang en --output-dir outputs/1_xlmr_seed42 --seed 42 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed42
python train.py --model xlmr --lang en --output-dir outputs/1_xlmr_seed43 --seed 43 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed43
python train.py --model xlmr --lang en --output-dir outputs/1_xlmr_seed44 --seed 44 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed44

## Evaluate zero-shot on the held-out Bengali test split

In [ ]:
%%bash
set -e
python evaluate.py --model-dir outputs/1_xlmr_seed42/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/1_xlmr_seed43/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/1_xlmr_seed44/final --lang bn --split test --seed 42

# metrics are tiny - push them across too
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed42 --metrics-only
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed43 --metrics-only
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment1 --run-dir outputs/1_xlmr_seed44 --metrics-only

## Summarise across seeds

In [ ]:
import glob, json, statistics as st

def mean_std(vals):
    m = st.mean(vals)
    s = st.stdev(vals) if len(vals) > 1 else 0.0
    return f"{m:.4f} +/- {s:.4f}"

tag = "1_xlmr"
runs = [json.load(open(p)) for p in sorted(glob.glob(f"outputs/{tag}_seed*/final/eval_test.json"))]
assert runs, f"{tag}: no results"

print(f"{tag:16s} n={len(runs)}")
for k in ("macro_f1", "qwk", "mae", "accuracy"):
    print(f"  {k:18s} {mean_std([r[k] for r in runs])}")
for k in ("upgrade_rate", "downgrade_rate", "error_rate", "mean_signed_error"):
    print(f"  {k:18s} {mean_std([r['direction'][k] for r in runs])}")

print("\nper-class F1 (mean across seeds):")
for cls in ("minimal", "mild", "moderate", "severe"):
    print(f"  {cls:9s} {mean_std([r[cls] for r in runs])}")

print("\nper-class P / R (mean across seeds):")
for cls in ("minimal", "mild", "moderate", "severe"):
    p_mean = st.mean([r["classification_report"][cls]["precision"] for r in runs])
    r_mean = st.mean([r["classification_report"][cls]["recall"] for r in runs])
    print(f"  {cls:9s} {p_mean:.4f} / {r_mean:.4f}")

In [ ]:
# Export metrics to the local machine. Checkpoints are large and live on the HF, this zip is just the small numbers.
import glob, zipfile

paths = sorted(
    glob.glob("outputs/1_xlmr_seed*/final/eval_test.json")
    + glob.glob("outputs/1_xlmr_seed*/train_metrics.json")
)
assert paths, "no results found - run the training and evaluation cells first"

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p)

print(f"zipped {len(paths)} files:")
for p in paths:
    print(" ", p)

from google.colab import files
files.download("results.zip")
print("downloaded results.zip")

In [ ]:
# What ended up on the HF. Expect 3 run directories (~3.3 GB total).
# Only the best checkpoint per run is kept; intermediate checkpoint-* folders are deliberately not uploaded.
from huggingface_hub import HfApi
api = HfApi()
info = api.repo_info(HF_REPO, files_metadata=True)
print(f"repo: {info.id}  (private={info.private})")
total = 0
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    size = getattr(s, "size", 0) or 0
    total += size
    print(f"  {size/1e6:8.1f} MB  {s.rfilename}")
print(f"  total {total/1e6:.1f} MB")

## Notes

- **Where the models live:** a **private** Hugging Face Hub model repo,
  `simonfrnd/nlp_experiment1`, set explicitly in the auth cell above and
  passed via `--repo-id` on every push. Each run writes
  `outputs/1_xlmr_seed<N>/final/` (best checkpoint by English validation macro-F1) and
  is pushed as `<run>/final/`, plus `train_metrics.json` and
  `val_predictions.npz`. Intermediate `checkpoint-*` folders stay on the VM only
  and are deliberately not uploaded.
- **Authentication:** set `HF_TOKEN` in Colab Secrets (key icon, left sidebar).
  The token is read from there and never printed. Create one at
  https://huggingface.co/settings/tokens with `write` scope.
- **Keep it private.** The source datasets state no licence, so model weights
  trained on them should not be published until that is settled.
- **Sizes:** Condition 1 (XLM-R) ~1.11 GB per seed. Three runs = ~3.3 GB, which
  fits in the free 100 GB private quota.
- **Expected runtime:** ~15-20 min per training run on a T4. Three runs = ~50 min.
  Because checkpoints are pushed after every run, a crash or idle-timeout costs
  at most one run.
- **Re-running after a runtime restart:** re-run all cells. Cell 4 re-clones,
  cell 5 re-authenticates, and already-pushed runs can be skipped by deleting
  their `python train.py` lines (or just re-run - results are deterministic
  per seed).
- **Downstream reuse:** Condition 3 (Translate-Test) and RQ2 (SHAP attributions)
  might reuse these trained weights directly.
- **Ablation:** `--no-class-weights` disables the weighted loss.